# Evaluating LLM Outputs using LLUMO API

This notebook evaluates the quality of LLM-generated responses based on key metrics such as confidence, relevance, and accuracy etc. The evaluation is performed using the LLUMO API.

Step 1: Import Required Libraries

In [40]:
import requests
import json
import time
import pandas as pd
from google.colab import userdata

Step 2: Define API Configuration


In [41]:
# API Authorization Header (Replace with a valid API key)
LLUMO_API_KEY = userdata.get("LLUMO_API_KEY")

headers = {
    "Authorization": f"Bearer {LLUMO_API_KEY}"
}

# API Endpoint for evaluation
LLUMO_ENDPOINT = "https://app.llumo.ai/api/create-eval-analytics"


Step 3: Load and Prepare Dataset


```
We begin by loading a dataset containing:

query: The input question or query.
ragContext: The context retrieved for the query.
ground_truth: The expected correct response.
output: The LLM-generated response.
```






In [ ]:
# Load dataset from an Excel file
# Ensure the dataset contains required columns: 'query', 'ragContext', 'ground_truth', 'output'
data = pd.read_excel("dataset.xlsx", usecols=["query", "ragContext", "ground_truth", "output"])

# Initialize columns to store evaluation results
metrics = ["confidence", "relevance", "accuracy"]
for metric in metrics:
    data[metric] = None  # Set default values as None
    data[f"{metric}_reason"] = None  # Column to store reasoning


Step 4: Process Each Query and Evaluate


In [ ]:
for indx, row in data.iterrows():
    query = row["query"]
    context = row["ragContext"]
    ground_truth = row["ground_truth"]
    llm_output = row["output"]

    # Construct request payload for API
    req_body = {
        "prompt": f"Give me an answer to the following query: {query}, using the given context: {context}",
        "input": {
            "query": query,
            "rag_context": context,
        },
        "output": llm_output,
        "analytics": metrics  # Specify evaluation metrics
    }

    try:
        # Send POST request to LLUMO API
        response = requests.post(url=LLUMO_ENDPOINT, json=req_body, headers=headers, timeout=18)
        response_json = response.json()

        # Extract data from response
        response_data = json.loads(response_json["data"]["data"])
        print(response_data)  # Debugging print statement

        # Store analytics scores and reasoning
        for metric in metrics:
            data.at[indx, metric] = response_data["analyticsScore"].get(metric, None)
            data.at[indx, f"{metric}_reason"] = response_data["reasoning"].get(metric, [])

        print(f"✅ Processed row {indx}")

    except Exception as e:
        print(f"❌ Error at row {indx}: {e}")
        continue


{'analyticsScore': {'confidence': 75, 'relevance': 85, 'accuracy': 90, 'context': 80, 'clarity': 95, 'overallScore': 85}, 'reasoning': {'confidence': ['The output is written in a declarative style, suggesting a degree of certainty.', 'However, it lacks explicit confidence indicators like strong adverbs or phrases expressing absolute certainty.', 'The response is concise and to the point, which can be interpreted as confident, but it could benefit from a more detailed explanation to enhance confidence.'], 'relevance': ["The output directly addresses the prompt's core question about the number of buttons on the remote.", 'It effectively summarizes the key discussion points from the provided transcript regarding the desired functionality and simplicity of the remote.', "The suggestion of essential buttons and a menu button aligns well with the meeting's conclusions."], 'accuracy': ['The output accurately reflects the consensus reached in the meeting regarding the number and type of button

Step 5: Display Final Evaluated Data

In [ ]:
# Display final dataset with results
print(data)

,query,ragContext,output,ground_truth,confidence,relevance,accuracy,confidence_reason,relevance_reason,accuracy_reason
0,How many buttons should be included in the rem...,Project Manager: Okay. Um welcome to our secon...,The remote control should include a limited nu...,The remote control should include the essentia...,75,85,90,"[The output is written in a declarative style,...",[The output directly addresses the prompt's co...,[The output accurately reflects the consensus ...
1,What are the main points discussed in the meet...,Project Manager: Hello.\nMarketing: Hey guys.\...,The main points discussed in the meeting inclu...,The main points of the meeting were to keep th...,75,85,90,[The output displays confidence by presenting ...,[All points in the output directly relate to t...,[The output correctly identifies key discussio...
2,What are the key takeaways from the meeting ab...,"Gareth Rogers: Good morning, and welcome to to...",The meeting revealed several key takeaways:\n\...,The meeting highlighted several key concerns a...,75,90,85,[The output demonstrates confidence through it...,[The output directly addresses all the key asp...,[The output accurately reflects the main point...
3,What are Dr. Frank Atherton's views on the use...,"Lynne Neagle AM: Good morning, everyone. Welco...",Dr. Atherton thinks it's definitely something ...,Dr. Atherton thinks it's something to consider...,75,85,90,[The output demonstrates confidence by stating...,[The output directly addresses the prompt's co...,[The output accurately summarizes Dr. Atherton...
4,I'm curious about the design of this remote co...,Project Manager: Okay. Uh first of all I'll st...,The special color of the buttons was intended ...,The special color of the buttons was meant to ...,75,85,90,[The output is written in a clear and concise ...,[The output directly answers the prompt's ques...,[The output correctly identifies the initial p...
